In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!tar -cf "/content/drive/MyDrive/gee_chips_2015_random20k.tar" \
-C "/content/drive/MyDrive/india_2015_images" \
"gee_chips_2015_random20k"

!cp "/content/drive/MyDrive/gee_chips_2015_random20k.tar" /content/

!rm -rf /content/chips_local
!mkdir -p /content/chips_local

!tar -xf "/content/gee_chips_2015_random20k.tar" \
-C /content/chips_local \
--strip-components=1

In [ ]:
import os, glob, re
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import resnet18

In [ ]:
# Paths to data and metadata
IMAGES_DIR = "/content/chips_local"
CLUSTERS_CSV = "/content/drive/MyDrive/IN2015_clusters_nightlights.csv"

CKPT_PATH = "/content/drive/MyDrive/models_india_2015/"
CKPT_PATH += "cnn_random20k_nightlights_proxy_resnet18_best.pt"

# Output file
OUT_CLUSTER_EMB_CSV = (
    "/content/drive/MyDrive/models_india_2015/"
    "india2015_cluster_embeddings_random20k_resnet18_proxy_nl.csv"
)

BATCH_SIZE = 128
NUM_WORKERS = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Load clusters and image table

In [ ]:
clusters = pd.read_csv(CLUSTERS_CSV)
clusters["cluster_id"] = clusters["cluster_id"].astype(int)

# Scan image files
image_paths = sorted(glob.glob(os.path.join(IMAGES_DIR, "*.png")))
rows = []
for p in image_paths:
    name = os.path.basename(p)
    m = re.search(r"c(\d+)_s(\d+)_lat([\d\.-]+)_lon([\d\.-]+)", name)
    if m:
        cid = int(m.group(1))
        rows.append({
            "cluster_id": cid,
            "image_path": p,
        })

df = pd.DataFrame(rows)

## Dataset ans Transforms

In [ ]:
tfm = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

class ImgDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        p = self.df.loc[idx, "image_path"]
        img = Image.open(p).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

In [ ]:
import torch.nn as nn
from torchvision.models import resnet18

# Number of classes for nightlights bins
N_CLASSES = 5

model = resnet18(weights=None)
model.fc = nn.Linear(512, N_CLASSES)

# Load checkpoint
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
state_dict = ckpt["state_dict"]

model.load_state_dict(state_dict)
model = model.to(DEVICE)
model.eval()

# Remove classification head to create an embedder
embedder = nn.Sequential(*list(model.children())[:-1])
embedder = embedder.to(DEVICE).eval()

## Extract embeddings

In [ ]:
ds = ImgDataset(df, transform=tfm)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)

all_emb = []
with torch.no_grad():
    for x in tqdm(loader, desc="Extracting embeddings"):
        x = x.to(DEVICE, non_blocking=True)
        z = embedder(x).squeeze(-1).squeeze(-1)
        all_emb.append(z.cpu().numpy())

emb = np.vstack(all_emb)

In [ ]:
emb_cols = [f"f{i}" for i in range(emb.shape[1])]
df_emb = df[["cluster_id"]].copy()
df_emb[emb_cols] = emb

cluster_emb = df_emb.groupby("cluster_id")[emb_cols].mean().reset_index()
cluster_emb.to_csv(OUT_CLUSTER_EMB_CSV, index=False)
print("Saved:", OUT_CLUSTER_EMB_CSV)